In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/ml_daily_features.parquet")

# ensuring date is DATETIME and year is INT
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["year"].astype(int)

df.head()
df.dtypes


site_id                    object
site_name                  object
pollutant                  object
unit                       object
date               datetime64[ns]
year                        int64
daily_median              float64
daily_mean                float64
roll7                     float64
roll30                    float64
norm_median               float64
delta_abs                 float64
delta_pct                 float64
z                         float64
exceed_pm25_who             int64
exceed_pm10_who             int64
dtype: object

In [8]:
df["pollutant"].unique()

array(['BSP', 'CO', 'DBT', 'NAN', 'NO2', 'O3', 'PM10', 'PM2.5', 'SIG05',
       'SIG60', 'SO2', 'SWD', 'SWS', 'VISIBILITYREDUCTION', 'VWD', 'VWS',
       'BPM2.5'], dtype=object)

In [ ]:
df["pollutant"] = (
    df["pollutant"]
    .str.upper()
    .str.replace(" ", "")
    .str.replace(".", "")
)

array(['BSP', 'CO', 'DBT', 'NAN', 'NO2', 'O3', 'PM10', 'PM2.5', 'SIG05',
       'SIG60', 'SO2', 'SWD', 'SWS', 'VISIBILITYREDUCTION', 'VWD', 'VWS',
       'BPM2.5'], dtype=object)

In [ ]:
df_pm25 = df[df["pollutant"] == "PM25"].copy()

df_pm25.shape
df_pm25["site_id"].nunique(), df_pm25["site_name"].nunique()

(18, 17)

In [ ]:
# Implement missing value logic, only on the PM2.5 subset
df_pm25["z_missing"] = df_pm25["z"].isna().astype(int)
df_pm25["delta_pct_missing"] = df_pm25["delta_pct"].isna().astype(int)
df_pm25["norm_missing"] = df_pm25["norm_median"].isna().astype(int)
df_pm25["roll30_missing"] = df_pm25["roll30"].isna().astype(int)

# Impute
# z: treat missing values as "no surprise"
df_pm25["z"] = df_pm25["z"].fillna(0)

# delta_pct: missing when norm_median == 0 -> treat as 0% change
df_pm25["delta_pct"] = df_pm25["delta_pct"].fillna(0)

# norm_median: fill with median norm for that site and pollutant
df_pm25["norm_median"] = (
    df_pm25
    .groupby(["site_id", "pollutant"])["norm_median"]
    .transform(lambda s: s.fillna(s.median()))
)

# delta_abs: recompute from daily_median - norm_median (this method is safer than filling directly)
df_pm25["delta_abs"] = df_pm25["daily_median"] - df_pm25["norm_median"]

# roll30: where missing, fall back to roll7
df_pm25["roll30"] = df_pm25["roll30"].fillna(df_pm25["roll7"])

c:\Users\Christopher\Desktop\Personal Projects\Project 1 Comparing Melbourne's Air Quality\mel-air-norms-now\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [5]:
df_pm25.isna().mean().sort_values(ascending=False).head(10)

delta_abs       0.015339
norm_median     0.015339
site_id         0.000000
site_name       0.000000
unit            0.000000
pollutant       0.000000
date            0.000000
year            0.000000
daily_mean      0.000000
daily_median    0.000000
dtype: float64